In [6]:
# ========== 导入：合成「技术文档 / 知识库」Markdown 生成器所需工具 ==========

# 导入标准库 os：读环境变量（例如 OPENAI_API_KEY）
import os
# 导入标准库 json：本练习后续若处理结构化数据可用（当前格先导入）
import json
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 导入 gradio：快速搭 Web UI（Blocks / Textbox / Button 等）
import gradio as gr
# 再次导入 json（与原文一致；重复 import 无害，逻辑不改）
import json
# 从 openai 导入 OpenAI 客户端：调用 Chat Completions API
from openai import OpenAI
# 导入标准库 re：用正则去掉模型可能包上的 ``` 代码围栏
import re


In [7]:
# ========== 环境准备：加载 .env，取出密钥，创建 OpenAI 客户端 ==========

# override=True：.env 中的值覆盖进程里已有的同名环境变量
load_dotenv(override=True)
# 从环境变量读取 OpenAI API Key（变量名必须保持 OPENAI_API_KEY）
api_key = os.getenv('OPENAI_API_KEY')
    
# 默认客户端：SDK 会自动使用环境里的 OPENAI_API_KEY（此处未显式传入 api_key）
client = OpenAI()


In [8]:
# ========== System Prompt：规定「只输出可检索的 Markdown 知识库文档」 ==========

# system_prompt：角色与格式硬约束；英文原文必须保留（翻译会改变模型行为）
system_prompt = """
    You are an expert technical writer and knowledge engineer.
    Your task is to generate well-structured Markdown (.md) documentation files that can be used as a knowledge base for a RAG.

    Follow these rules carefully:
    1. Write the content in clear, concise Markdown format.
    2. Use appropriate Markdown headers (#, ##, ###) to structure the document.
    3. Include lists, tables, or code blocks only when necessary.
    4. Keep each document self-contained and focused on a single topic.
    5. Do not include any text outside the Markdown content (no explanations, no code fences).
    6. The style should be factual, structured, and helpful for machine retrieval.
    7. Use consistent tone and terminology across sections.
    """


In [9]:
# ========== User Prompt 工厂：把 topic / kb_type 填进固定英文模板 ==========

def create_kb_prompt(topic, kb_type="tutorial"):
    # 返回 f-string：发给模型的 user 消息；模板英文与默认 kb_type 保持原样
    return f"""
    Generate a comprehensive Markdown document for the following technical topic.
    Topic: {topic}
    Document Type: {kb_type}
    The document should include structured sections, concise explanations, and clear formatting suitable for a technical knowledge base.
    """


In [10]:
# ========== 核心调用：system + user → Chat Completions → 清洗围栏 ==========

def generate_markdown_doc(topic, kb_type="tutorial"):
    
    # 用工厂函数拼出本轮 user prompt（含主题与文档类型）
    user_prompt = create_kb_prompt(topic, kb_type)
    # messages：标准 Chat 角色列表；system 定规矩，user 给任务
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    
    # 调用云端 Chat Completions；model / temperature 等参数保持原文
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        temperature=0.7
    )
    # 取出第一条 choice 的文本，并去掉首尾空白
    markdown_output = response.choices[0].message.content.strip()
    # 若模型仍包了 ```md ... ```，用正则剥掉开头围栏（模式字符串保持原样）
    markdown_output = re.sub(r'^```[a-z]*\\s*', '', markdown_output, flags=re.MULTILINE)
    # 再剥掉结尾的 ``` 围栏
    markdown_output = re.sub(r'\\s*```$', '', markdown_output, flags=re.MULTILINE)
    # 返回纯 Markdown 字符串，供 Gradio 文本框展示
    return markdown_output


In [ ]:
# ========== Gradio UI：左侧输入主题/类型，右侧展示生成的 Markdown ==========

def create_kb_gradio_interface():
    # Soft 主题的 Blocks 应用容器；with 块内声明组件树
    with gr.Blocks(theme=gr.themes.Soft()) as app:
        # 页面标题（UI 字符串保持英文原样，避免改交互文案）
        gr.Markdown("## Technical Knowledge Base Generator")

        # 一行两列：左输入、右输出
        with gr.Row():
            with gr.Column():
                # 主题输入框：lines=2 允许多行粘贴较长主题
                topic_input = gr.Textbox(
                    label="Technical Topic",
                    placeholder="e.g., Building a RAG pipeline with LangChain...",
                    lines=2
                )
                # 文档类型单选：choices / 默认 value 必须与原逻辑一致
                kb_type_input = gr.Radio(
                    label="Document Type",
                    choices=["Overview", "FAQ", "Use Case"],
                    value="FAQ"
                )
                # 主按钮：点击后触发 generate_markdown_doc
                generate_button = gr.Button("Generate Markdown Document", variant="primary")

            with gr.Column():
                # 只读大文本框：展示生成结果（interactive=False）
                output_md = gr.Textbox(
                    label="Generated Markdown Content",
                    lines=25,
                    interactive=False,
                    placeholder="Generated Markdown will appear here..."
                )

        # 绑定点击事件：inputs → fn → outputs；api_name 供 Gradio API 调用
        generate_button.click(
            fn=generate_markdown_doc,
            inputs=[topic_input, kb_type_input],
            outputs=[output_md],
            api_name="generate_kb_doc"
        )

    # 返回构建好的 Blocks 应用对象
    return app


In [ ]:
# ========== 启动应用：创建界面并 launch（debug + 公共 share 链接） ==========

# 调用工厂函数得到 Gradio Blocks 实例
app = create_kb_gradio_interface()
# launch：本地起服务；debug=True 打印更多错误；share=True 尝试生成公网临时链接
app.launch(debug=True, share=True)
